# Verify GRID pretraining preprocessing end-to-end

Runs the exact same preprocessing pipeline `scripts/pretrain_landmark_grid.py`'s
`main()` runs, step by step, against real data: build/load `grid_manifest`,
split it by held-out speaker, compute/load pixel stats from the train
speakers only, build the train/val `GridWordSegmentDataset`s, and display
a few real samples from each split -- so a full training run isn't the
first time any of this is actually looked at.

**What "looks right" means:**
- `grid_manifest` and both datasets build without errors, with a sane
  number of rows/word segments (not suspiciously small or empty).
- Pixel stats (`mean`, `std`) are plausible grayscale values (mean
  somewhere in [0, 255], std nonzero and not enormous).
- Train and val speakers are completely disjoint (the whole point of
  `_split_manifest_by_speaker`).
- Displayed samples: each one's word matches what the patches visually
  look like (not blank, not obviously misaligned), across BOTH splits.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent  # adjust if running from somewhere else
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# TODO: fill in correct paths (same convention as notebook 01)
PROJECT_NAME = "your_project_name"
DATASET_ROOT = Path(f"/scratch/{PROJECT_NAME}/datasets")

grid_all_path = "kaggle_lipnet/datasets/jedidiahangekouakou/grid-corpus-dataset-for-training-lipnet/versions/1/data"
GRID_ROOT = DATASET_ROOT / grid_all_path
GRID_LANDMARKS_ROOT = DATASET_ROOT / "grid_landmarks"
AUDIO_OUTPUT_DIR = DATASET_ROOT / "extracted_audio"

LIMIT = 200  # a smoke-test-sized slice, not the full ~33,000 clips
SEED = 42
VAL_SPEAKER_FRACTION = 0.1
PIXEL_STATS_FRAMES_PER_VIDEO = 10

## Step 1: build/load `grid_manifest` (same call `main()` makes)

In [ ]:
from fusion_avsr.data.manifest_builder import build_grid_manifest, load_or_build_manifest
from fusion_avsr.data.paths import MANIFEST_DIR

grid_manifest = load_or_build_manifest(
    MANIFEST_DIR / "grid_manifest.csv", build_grid_manifest,
    grid_root=GRID_ROOT, landmarks_root=GRID_LANDMARKS_ROOT,
    audio_output_dir=AUDIO_OUTPUT_DIR, limit=LIMIT,
)
print("grid_manifest:", grid_manifest.shape)
grid_manifest.head()

## Step 2: split the manifest by held-out speaker

Uses the exact same `_split_manifest_by_speaker` function
`scripts/pretrain_landmark_grid.py` uses, imported directly from it so
this notebook can never silently drift from what training actually does.
Splitting BEFORE pixel stats are computed means train speakers never
leak into the normalization stats used to train on them.

In [ ]:
from scripts.pretrain_landmark_grid import _split_manifest_by_speaker

train_manifest, val_manifest = _split_manifest_by_speaker(grid_manifest, VAL_SPEAKER_FRACTION, SEED)
print(f"train_manifest: {train_manifest.shape}, val_manifest: {val_manifest.shape}")

## Step 3: compute/load pixel stats (train speakers only)

In [ ]:
from fusion_avsr.models.landmark.normalization import load_or_compute_pixel_stats

pixel_mean, pixel_std = load_or_compute_pixel_stats(
    MANIFEST_DIR / "grid_pixel_stats.json",
    video_paths=train_manifest["video_path"].tolist(),
    frames_per_video=PIXEL_STATS_FRAMES_PER_VIDEO,
    seed=SEED,
)
print(f"pixel_mean={pixel_mean:.4f}, pixel_std={pixel_std:.4f}")
assert 0 <= pixel_mean <= 255, "mean should be a plausible grayscale pixel value"
assert pixel_std > 0, "std should be nonzero (real variation across frames)"

## Step 4: build the train and val datasets separately

Each dataset is built directly from its own manifest split -- no
combined dataset + index-based `Subset` needed, since the split already
happened at the manifest level in Step 2. `val_dataset` reuses
`train_dataset.vocabulary` so class indices line up across both splits.

In [ ]:
from fusion_avsr.models.landmark.dataset import GridWordSegmentDataset

train_dataset = GridWordSegmentDataset(
    grid_root=GRID_ROOT,
    grid_manifest=train_manifest,
    pixel_mean=pixel_mean,
    pixel_std=pixel_std,
    limit=LIMIT,
)
val_dataset = GridWordSegmentDataset(
    grid_root=GRID_ROOT,
    grid_manifest=val_manifest,
    pixel_mean=pixel_mean,
    pixel_std=pixel_std,
    vocabulary=train_dataset.vocabulary,
    limit=LIMIT,
)
print(f"train: {len(train_dataset)} word segments, val: {len(val_dataset)} word segments")
print(f"vocabulary: {len(train_dataset.vocabulary)} words")

## Sanity check: train/val speakers are completely disjoint

In [ ]:
train_speakers = set(train_manifest["sample_id"].str.split("_", n=1).str[0])
val_speakers = set(val_manifest["sample_id"].str.split("_", n=1).str[0])

print(f"train speakers ({len(train_speakers)}): {sorted(train_speakers)}")
print(f"val speakers ({len(val_speakers)}): {sorted(val_speakers)}")

overlap = train_speakers & val_speakers
print(f"overlap: {overlap or 'none'}")
assert not overlap, "train and val must never share a speaker"

## Step 5: display a few real samples from each split

Patches are shown DEnormalized (`patches * pixel_std + pixel_mean`) back
to a viewable ``[0, 255]`` grayscale range -- `dataset[i]` itself returns
already pixel-normalized tensors, which wouldn't look like a face if
plotted directly.

In [ ]:
import matplotlib.pyplot as plt

from fusion_avsr.models.landmark.lrlp import NUM_LRLPS

vocabulary_inv = {index: word for word, index in train_dataset.vocabulary.items()}
NUM_SAMPLES_PER_SPLIT = 3


def show_sample(dataset, index, title_prefix):
    patches, _aligned_coords, label = dataset[index]
    word = vocabulary_inv[label]
    num_frames = patches.shape[1]
    mid_frame = num_frames // 2

    denorm = (patches.numpy() * pixel_std + pixel_mean).clip(0, 255).astype("uint8")

    fig, axes = plt.subplots(4, 10, figsize=(14, 6))
    fig.suptitle(f"{title_prefix}: word={word!r}, T={num_frames} (frame {mid_frame} shown)")
    for k, ax in enumerate(axes.ravel()):
        if k < NUM_LRLPS:
            ax.imshow(denorm[k, mid_frame], cmap="gray")
            ax.set_xticks([]); ax.set_yticks([])
        else:
            ax.axis("off")
    plt.tight_layout()
    plt.show()

### Train samples

In [ ]:
import random

rng = random.Random(SEED)
for i in rng.sample(range(len(train_dataset)), k=min(NUM_SAMPLES_PER_SPLIT, len(train_dataset))):
    show_sample(train_dataset, i, "train")

### Val samples

In [ ]:
rng = random.Random(SEED)
for i in rng.sample(range(len(val_dataset)), k=min(NUM_SAMPLES_PER_SPLIT, len(val_dataset))):
    show_sample(val_dataset, i, "val")

## Checklist

- [ ] `grid_manifest`, `train_dataset`, and `val_dataset` all built
      without errors, with a sane (non-tiny, non-empty) number of rows.
- [ ] `pixel_mean`/`pixel_std` are plausible grayscale values (asserted
      above).
- [ ] Train/val speaker sets are completely disjoint (asserted above).
- [ ] Every displayed sample's patches visibly look like a mouth/face
      region -- not blank, not garbage -- for BOTH train and val.
- [ ] Once this all looks right with `LIMIT` set, consider re-running
      with `LIMIT = None` before trusting a full training run (slower,
      since pixel stats + manifest building scale with the real dataset
      size).